# Operational analysis

All records in this notebook are synthetic. The period is June 2026, with May as the comparison. These are recorded cash totals, not profit. Tenant grades describe owner review priority and do not prove fault.

In [1]:
import json
import sqlite3
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path
import joblib
from kos_models.tenant_model import score_tenants
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent.parent
histories = [json.loads(line) for line in (root / "services/models/data/histories.jsonl").open()]
records = [{**activity, "tenantId": tenant["tenantId"]} for tenant in histories for activity in tenant["activities"]]
as_of = "2026-06-30"
period_start = "2026-06-01"
visible = [row for row in records if (row["completedDate"] or row["occurrenceDate"] or "9999") <= as_of]
len(histories), len(visible)

(600, 35183)

## Pending activities

The app counts planned/in-progress work or draft money as needing attention. A rent row can remain planned even after payment. A future-dated settlement is not yet paid at this snapshot.

In [2]:
pending = [row for row in visible if row["workStatus"] in ("planned", "in_progress") or row["paymentStatus"] == "draft" or (row["paymentDate"] is not None and row["paymentDate"] > as_of)]
overdue = [row for row in visible if row["categoryId"] == "activity-category-room-rent" and row["occurrenceDate"] < as_of and (row["paymentStatus"] != "settled" or row["paymentDate"] is None or row["paymentDate"] > as_of)]
pending_categories = dict(Counter(row["categoryId"] for row in pending))
{"pending": len(pending), "overdueRentOccurrences": len(overdue), "pendingByCategory": pending_categories}

{'pending': 10477,
 'overdueRentOccurrences': 36,
 'pendingByCategory': {'activity-category-room-rent': 10451,
  'activity-category-routine-cleaning': 21,
  'activity-category-corrective-repairs': 4,
  'activity-category-extra-cleaning': 1}}

## Changes in settled cash

Filter by actual payment date, not work date. Shared expenses stay in All rooms. No balance or expected rent is added to settled cash.

In [3]:
monthly = defaultdict(lambda: {"income": 0, "expenses": 0})
for row in visible:
    if row["paymentStatus"] == "settled" and row["paymentDate"] and row["paymentDate"] <= as_of and row["amount"] is not None:
        key = "income" if row["financialKind"] == "income" else "expenses"
        monthly[row["paymentDate"][:7]][key] += row["amount"]
for values in monthly.values():
    values["net"] = values["income"] - values["expenses"]
comparison = {key: monthly["2026-06"][key] - monthly["2026-05"][key] for key in ("income", "expenses", "net")}
{"May": monthly["2026-05"], "June": monthly["2026-06"], "JuneMinusMay": comparison}

{'May': {'income': 1088600000, 'expenses': 611800000, 'net': 476800000},
 'June': {'income': 1092600000, 'expenses': 394250000, 'net': 698350000},
 'JuneMinusMay': {'income': 4000000, 'expenses': -217550000, 'net': 221550000}}

In [4]:
connection = sqlite3.connect(":memory:")
connection.execute("create table ledger (payment_date text, payment_status text, kind text, amount integer)")
connection.executemany("insert into ledger values (?, ?, ?, ?)", [(r["paymentDate"], r["paymentStatus"], r["financialKind"], r["amount"]) for r in visible])
reference = dict(connection.execute("select kind, sum(amount) from ledger where payment_status = 'settled' and payment_date between '2026-06-01' and '2026-06-30' group by kind").fetchall())
assert monthly["2026-06"]["income"] == reference.get("income", 0)
assert monthly["2026-06"]["expenses"] == reference.get("expense", 0)
connection.close()
reference

{'expense': 394250000, 'income': 1092600000}

## Tenant review priorities

Use the saved model and the serving feature order. The financial filter does not change the rolling history window.

In [5]:
artifact = joblib.load(root / "services/models/artifacts/tenant-detector.joblib")
snapshots = [json.loads(line) for line in (root / "services/models/data/snapshots.jsonl").open()]
latest = [row for row in snapshots if row["asOfDate"] == as_of]
eligible = [row for row in latest if not row["eligibilityReasons"]]
scores, grades = score_tenants(artifact, eligible)
priorities = [{"tenantId": row["tenantId"], "grade": str(grade), "score": float(score), "windowStart": row["windowStart"], "facts": row["facts"]} for row, score, grade in zip(eligible, scores, grades, strict=True)]
priorities.sort(key=lambda row: ({"C": 0, "B": 1, "A": 2}[row["grade"]], -row["score"], row["tenantId"]))
{"eligible": len(priorities), "notEnoughData": len(latest) - len(priorities), "gradeCounts": dict(Counter(row["grade"] for row in priorities)), "firstEight": priorities[:8]}

{'eligible': 580,
 'notEnoughData': 20,
 'gradeCounts': {'C': 45, 'B': 34, 'A': 501},
 'firstEight': [{'tenantId': 'synthetic-0116',
   'grade': 'C',
   'score': 1.9998209884635842,
   'windowStart': '2025-07-01',
   'facts': [{'kind': 'repairs90',
     'value': 12,
     'activityIds': ['synthetic-0116-activity-90',
      'synthetic-0116-activity-91',
      'synthetic-0116-activity-92',
      'synthetic-0116-activity-93',
      'synthetic-0116-activity-101',
      'synthetic-0116-activity-102',
      'synthetic-0116-activity-103',
      'synthetic-0116-activity-104',
      'synthetic-0116-activity-113',
      'synthetic-0116-activity-114',
      'synthetic-0116-activity-115',
      'synthetic-0116-activity-116']},
    {'kind': 'extraCleaning90',
     'value': 9,
     'activityIds': ['synthetic-0116-activity-94',
      'synthetic-0116-activity-95',
      'synthetic-0116-activity-96',
      'synthetic-0116-activity-97',
      'synthetic-0116-activity-105',
      'synthetic-0116-activity-

In [6]:
expense_categories = Counter()
for row in visible:
    if row["financialKind"] == "expense" and row["paymentStatus"] == "settled" and row["paymentDate"] and period_start <= row["paymentDate"] <= as_of:
        expense_categories[row["categoryId"]] += row["amount"] or 0
recommendations = [f"Review {len(overdue)} overdue rent occurrences as of {as_of}; inspect the source records before follow-up."]
if expense_categories:
    category, amount = expense_categories.most_common(1)[0]
    recommendations.append(f"Inspect {category}, the largest recorded expense category in {period_start} through {as_of}: Rp {amount:,}.")
recommendations.append("Review the first eight saved tenant priorities with their facts. The model is experimental; do not infer tenant fault from the grade.")
report = {"synthetic": True, "periodStart": period_start, "periodEnd": as_of, "pending": len(pending), "overdueRentOccurrences": len(overdue), "cash": dict(monthly), "change": comparison, "modelVersion": artifact["modelVersion"], "priorities": priorities[:8], "recommendations": recommendations}
(root / "analysis/report.json").write_text(json.dumps(report, indent=2) + "\n")
recommendations

['Review 36 overdue rent occurrences as of 2026-06-30; inspect the source records before follow-up.',
 'Inspect activity-category-routine-cleaning, the largest recorded expense category in 2026-06-01 through 2026-06-30: Rp 338,390,000.',
 'Review the first eight saved tenant priorities with their facts. The model is experimental; do not infer tenant fault from the grade.']